In [ ]:
#--- PHASE 10.9: Reset Stable Checkpoint (Safe) ---
import os
import time
import shutil
import torch

stable_ckpt = "citywide_stpignn_checkpoint_STABLE.pt"
best_ckpt = "citywide_stpignn_best.pt"

# 1) Backup old stable checkpoint
if os.path.exists(stable_ckpt):
    backup_path = f"{stable_ckpt}.bak_{int(time.time())}"
    shutil.copy2(stable_ckpt, backup_path)
    print(f"Backed up existing stable checkpoint -> {backup_path}")

# 2) Optionally initialize model from best weights
if os.path.exists(best_ckpt):
    best = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(best["state_dict"])
    print(f"Loaded model weights from {best_ckpt}")
else:
    print("Best checkpoint not found; using current in-memory model weights.")

# 3) Reset optimizer + scaler state (fresh training dynamics)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-2)
scaler_amp = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# 4) Write fresh stable checkpoint
fresh_payload = {
    "state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scaler_state_dict": scaler_amp.state_dict(),
    "epoch": 1,
    "step": 0,
    "best_val_mse": float("inf"),
    "timestamp": time.ctime(),
    "note": "fresh reset checkpoint",
}
torch.save(fresh_payload, stable_ckpt)
print(f"Fresh stable checkpoint written -> {stable_ckpt}")